In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t015/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory interface traffic drop alert). It occurred between 2026-03-16 05:48:49 and 2026-03-16 05:52:50 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/api/v1/inventory/{productId}". The user is requesting an analysis of the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口流量下跌告警', {'start': '2026-03-16T05:48:49.267476+08:00', 'end': '2026-03-16T05:52:50+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory接口流量下跌告警<alert_event event_id="b4b6e1203c859a39ced4b16bdbc7a4fb" trans_id="qpm4khc96d4udbm0jhd32qljd5" rule_id="b7853e06-54b7-4409-9eea-9bb82fbfb252" rule_name="inventory接口流量下跌告警" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t015/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs
available_kpis = metrics_data['metric'].unique()
available_kpis

```
Out[2]:
```
The available KPIs in the metrics data include various node-level, deployment-level, cluster-level, and application-level metrics. Relevant KPIs for analyzing the traffic drop issue include `request_count`, `error_count`, `error_rate`, `latency`, and `slow`. These KPIs can help identify changes in traffic volume, errors, and performance issues related to the inventory operation.

The original code execution output of IPython Kernel is also provided below for reference:

['node_not_ready_status' 'node_memory_usage_rate' 'node_disk_usage_rate'
 'node_pod_running_count' 'node_cpu_usage_rate' 'node_ready_status'
 'deployment_desired_replicas' 'deployment_available_replicas'
 'deployment_ready_replicas' 'deployment_availability_rate'
 'deployment_memory_usage_total' 'deployment_cpu_usage_vs_limits'
 'deployment_cpu_usage_vs_requests' 'deployment_cpu_usage_total'
 'cluster_node_count' 'cluster_failed_node_count'
 'apiserver_concurrent_connections' 'cluster_number_of_not_ready_pods'
 'apiserver_non_2xx_read_request_qps' 'cluster_number_of_pod'
 'apiserver_non_2xx_write_request_qps' 'apiserver_requests_per_second'
 'namespace_number_of_not_ready_pods' 'namespace_number_of_pods'
 'error_count' 'slow_count' 'request_count' 'avg_request_latency_seconds'
 'arms_jvm_gc_total' 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'error' 'slow' 'error_rate']```
```

In [ ]:
# In[3]:

# Filter metrics data for the relevant KPIs
relevant_kpis = ['request_count', 'error_count', 'error_rate', 'latency', 'slow']
filtered_metrics = metrics_data[metrics_data['metric'].isin(relevant_kpis)].copy()

# Ensure 'value' column is numeric for calculations
filtered_metrics['value'] = pd.to_numeric(filtered_metrics['value'], errors='coerce')

# Calculate global P95 threshold for each KPI
p95_thresholds = filtered_metrics.groupby('metric')['value'].quantile(0.95)

p95_thresholds

```
Out[3]:
```
The global P95 thresholds for the relevant KPIs across all components are as follows:
- `error_count`: 592.75
- `error_rate`: 1.0
- `latency`: 0.047845 seconds
- `request_count`: 2064.7
- `slow`: 158.28

These thresholds represent the 95th percentile values for each KPI and can be used to identify anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

metric
error_count       592.750000
error_rate          1.000000
latency             0.047845
request_count    2064.700000
slow              158.284483
Name: value, dtype: float64```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Convert the alert window to timestamps in microseconds
timezone = pytz.timezone('Asia/Shanghai')
start_time = int(datetime.strptime(alert_window['start'], '%Y-%m-%dT%H:%M:%S.%f%z').timestamp() * 1_000_000)
end_time = int(datetime.strptime(alert_window['end'], '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)

# Filter metrics data within the time window
filtered_metrics['time'] = pd.to_numeric(filtered_metrics['time'], errors='coerce')
time_filtered_metrics = filtered_metrics[(filtered_metrics['time'] >= start_time) & (filtered_metrics['time'] <= end_time)]

# Identify anomalies where values exceed the global P95 thresholds
anomalies = time_filtered_metrics[time_filtered_metrics.apply(
    lambda row: row['value'] > p95_thresholds[row['metric']], axis=1
)]

anomalies

```
Out[4]:
```
Within the time window of 2026-03-16 05:48:49 to 2026-03-16 05:52:50 (UTC+8), anomalies were detected in the metrics data where values exceeded the global P95 thresholds. These anomalies include high `request_count` for the `frontend-proxy::ingress` operation, high `latency` for the `inventory` service, and high `slow` counts for the `cart` service. These anomalies suggest potential issues in these components during the alert period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set                         entity_id              entity_name         metric        value                            metric_set_id service
54881  1773611389000000    apm       apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc  frontend-proxy::ingress  request_count  2639.000000  apm@metric_set@apm.metric.apm.operation    None
54882  1773611419000000    apm       apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc  frontend-proxy::ingress  request_count  4746.000000  apm@metric_set@apm.metric.apm.operation    None
54883  1773611449000000    apm       apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc  frontend-proxy::ingress  request_count  5909.000000  apm@metric_set@apm.metric.apm.operation    None
54884  1773611479000000    apm       apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc  frontend-proxy::ingress  request_count  5734.000000  apm@metric_set@apm.metric.apm.operation    None
54885  1773611509000000    apm       apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc  frontend-proxy::ingress  request_count  5677.000000  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...                 ...                               ...                      ...            ...          ...                                      ...     ...
77563  1773611539000000    apm  apm.service.legacy                                                       cart           slow   193.000000                                     None    None
77599  1773611539000000    apm  apm.service.legacy                                                  inventory        latency     0.527870                                     None    None
77618  1773611539000000    apm  apm.service.legacy                                             recommendation        latency     0.050865                                     None    None
77638  1773611569000000    apm  apm.service.legacy                                                       cart           slow   193.000000                                     None    None
77674  1773611569000000    apm  apm.service.legacy                                                  inventory        latency     0.425890                                     None    None

[79 rows x 9 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t015/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime' and 'endTime' are numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Convert the time window to nanoseconds for filtering
start_time_ns = start_time * 1_000  # Convert microseconds to nanoseconds
end_time_ns = end_time * 1_000  # Convert microseconds to nanoseconds

# Filter traces within the time window
time_filtered_traces = traces_data[
    (traces_data['startTime'] >= start_time_ns) & (traces_data['endTime'] <= end_time_ns)
]

# Filter traces involving the 'inventory' service
inventory_traces = time_filtered_traces[time_filtered_traces['serviceName'].str.contains('inventory', case=False, na=False)]

# Analyze the call chain to determine the most downstream faulty service
downstream_faulty_services = inventory_traces[inventory_traces['statusCode'] != 0]

downstream_faulty_services

```
Out[5]:
```
Within the time window, traces involving the `inventory` service revealed downstream faults. These include SQL-related issues such as `SELECT inventory.inventory` and exceptions in `HikariDataSource.getConnection`. The most downstream faulty service appears to be the database accessed by the `inventory` service, as it encountered SQL exceptions during operations. This indicates potential database-related issues as the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                        spanName  ...  __topic__      __tag__:__pack_id__                                             events          ip links
400611  68e852888b990c7702e0cf4118f8bd38  627f4f0d6069cf55                      2             /actuator/health/**  ...             741E040EA4577B7C-6ECCA6                                               None   10.0.2.36  None
401538  284603583c0f0b0dfb4c3436b198734e  5d9bf8048dd7b407                      2             /actuator/health/**  ...             14476876A23D532A-6DE3BA                                               None  10.0.2.123  None
401543  d2c0b69789c96f9160e5f3bf26377ee1  6b6bcb0d7278696d  a3f9c4a3deaa21e6    3      SELECT inventory.inventory  ...             40D5E0D3FCBB6C24-6DDBD3                                               None   10.0.2.36  None
401544  0209d802710d7d057e5353c0260c1075  1ef0c3800f68a4da  607af68eebf15552    3      SELECT inventory.inventory  ...             40D5E0D3FCBB6C24-6DDBD3                                               None   10.0.2.36  None
402188  0900182faba9cd93783793c70c26967c  2f32ec3eb13b3fbd  95d07d2434d68418    3      SELECT inventory.inventory  ...             EB22547ADC249BDD-6C44AD                                               None  10.0.2.123  None
402189  14cc772b48b7953831aadcde03632ae2  30bcecc4031fb3ad  5b538b7e464adf25    3      SELECT inventory.inventory  ...             EB22547ADC249BDD-6C44AD                                               None  10.0.2.123  None
404149  1d3bb0a5adeb3c6a5f691ec7023880a4  0f42a02b1067fc63                      2             /actuator/health/**  ...             3930B19B26CDB367-6CE565                                               None   10.0.2.36  None
476205  f3ca4eb493b9f702bcc76e1d11699a08  76f92fc5284e7f49                      1  HikariDataSource.getConnection  ...             6BF1C380A76046DD-6E6F15  [{"attributes":{"exception.type":"java.sql.SQL...   10.0.2.36  None
476263  580d0ee79acc2d0c4717c18333796b43  71e7d67d63f2b0cc                      1  HikariDataSource.getConnection  ...             D6DB7BF7FD0380AD-6D8F63  [{"attributes":{"exception.type":"java.sql.SQL...  10.0.2.123  None

[9 rows x 21 columns]```
```